# Notebook 15 operational recovery runtime
Purpose: repair one orchestration API name in memory. Inputs: original immutable study binding/plan/clock plus a source-bound recovery sidecar. Outputs: existing scientific checkpoints and clearly labeled solver preflight receipts. Import only the `ahr-export` definition cell. No training runs during import. Runtime: definitions only; launches retain the original absolute deadline. Original 15/16 sources are unchanged.

In [ ]:
"""Operational API recovery only; preserve original scientific sources, plan and clock."""
from pathlib import Path
import copy, hashlib, inspect, json, time, types
import nbformat
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json, digest_file
from fmm_jepa.food101_training import run_writer
VERSION='notebook15-operational-api-recovery/v1'
RUNTIME='experiments/15_operational_recovery/runtime.ipynb'
LAUNCH='experiments/15_operational_recovery/launch.ipynb'
API={'workflow.run_worker':['root','run_id','worker'],'workflow.verify_binding':['root','artifact','binding'],
     'workflow.bootstrap_source':['root'],'workflow.spawn':['root','artifact','session','expression','path'],
     'evaluation.fastprobe_smoke':['root','artifact']}

def _read(path):return json.loads(Path(path).read_text())
def _canonical(value):return json.loads(json.dumps(value,sort_keys=True,allow_nan=False))
def _hash(value):return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def export_hash(path):
    cells=[c.source for c in nbformat.read(Path(path),4).cells if c.cell_type=='code' and 'ahr-export'in c.metadata.get('tags',[])]
    if not cells:raise ValueError('Recovery notebook has no explicitly tagged export definitions')
    return hashlib.sha256('\n'.join(cells).encode()).hexdigest()
def code_hash(path):
    return _hash([c.source for c in nbformat.read(Path(path),4).cells if c.cell_type=='code'])
def _api(root,workflow):
    evaluator=workflow.load(root,'evaluation');actual={}
    for key,expected in API.items():
        owner,name=key.split('.');module=workflow if owner=='workflow'else evaluator
        function=getattr(module,name);actual[key]=list(inspect.signature(function).parameters)
        if actual[key]!=expected:raise ValueError('Recovery API footprint changed: '+key)
    if hasattr(evaluator,'verify_evaluation_smoke') and not getattr(evaluator.verify_evaluation_smoke,'_operational_recovery_adapter',False):
        raise ValueError('Unexpected existing evaluation method; refuse alternate recovery')
    return actual

def paths(root,run_id):
    root=Path(root).resolve();run_id=''.join(str(run_id))
    if not run_id or any(not(c.isalnum()or c in '-_')for c in run_id):raise ValueError('Safe existing run ID required')
    artifact=root/'artifacts/notebook_15'/run_id
    return root,artifact,artifact/'operational_recovery_v1.json'

def create_binding(root,workflow,run_id):
    root,artifact,path=paths(root,run_id);original=artifact/'study_binding.json'
    if not original.exists():raise FileNotFoundError('Recovery requires the existing original study binding')
    binding=_read(original);workflow.verify_binding(root,artifact,binding)
    workflow.deadline_guard(binding['clock']['deadline_unix'])
    receipt={'schema':VERSION,'run_id':run_id,'original_binding_sha256':digest_file(original),
      'original_clock':copy.deepcopy(binding['clock']),'original_source_hashes':copy.deepcopy(binding['source_hashes']),
      'original_consumer_hash':binding['consumer_hash'],'reference_plan_sha256':binding['reference_plan_sha256'],
      'runtime_notebook':RUNTIME,'runtime_export_sha256':export_hash(root/RUNTIME),
      'launch_notebook':LAUNCH,'launch_code_sha256':code_hash(root/LAUNCH),'api_footprint':_api(root,workflow),
      'repair':'In-memory verify_evaluation_smoke adapter delegates to existing fastprobe_smoke; original scientific code/config/plan/deadline retained',
      'checkpoint_policy':'Original stage full-state checkpoints and latest-two/best policy unchanged',
      'resume':'Restart kernel and Run All in '+LAUNCH+' with the same FMM15_RUN_ID before the original deadline',
      'sessions':['fmm15-'+run_id,'fmm15-'+run_id+'-worker0','fmm15-'+run_id+'-worker1','fmm15-'+run_id+'-deadline'],
      'operator_notebooks':'Original paths receive _recovery_v1 suffix; original notebooks retained',
      'official_test':'NOT_RUN'}
    with run_writer(artifact/'operational_recovery_binding_writer'):
        if path.exists():
            if _read(path)!=receipt:raise ValueError('Existing immutable recovery binding differs')
        else:atomic_json(path,receipt)
    return receipt

def verify_recovery(root,workflow,run_id):
    root,artifact,path=paths(root,run_id)
    if not path.exists():raise FileNotFoundError('Recovery sidecar absent; review and create it explicitly')
    receipt=_read(path);binding=_read(artifact/'study_binding.json')
    if receipt.get('schema')!=VERSION or receipt.get('run_id')!=run_id:raise ValueError('Recovery schema/run ID differs')
    checks=[receipt['original_binding_sha256']==digest_file(artifact/'study_binding.json'),receipt['original_clock']==binding['clock'],
      receipt['original_source_hashes']==binding['source_hashes'],receipt['original_consumer_hash']==binding['consumer_hash'],
      receipt['reference_plan_sha256']==binding['reference_plan_sha256'],receipt['runtime_notebook']==RUNTIME,
      receipt['runtime_export_sha256']==export_hash(root/RUNTIME),receipt['launch_notebook']==LAUNCH,
      receipt['launch_code_sha256']==code_hash(root/LAUNCH),receipt['api_footprint']==_api(root,workflow)]
    if not all(checks):raise ValueError('Recovery sidecar/source/original binding changed')
    if _read(artifact/'budget_clock.json')!=receipt['original_clock']:raise ValueError('Original absolute clock changed')
    return receipt

def install(root,workflow,run_id):
    root,artifact,path=paths(root,run_id);receipt=verify_recovery(root,workflow,run_id)
    installed=getattr(workflow,'_operational_recovery',None)
    if installed is not None:
        if installed!={'run_id':run_id,'receipt_hash':_hash(receipt)}:raise ValueError('Different recovery already installed')
        return receipt
    original_verify=workflow.verify_binding;original_bootstrap=workflow.bootstrap_source;original_spawn=workflow.spawn
    evaluator=workflow.load(root,'evaluation')
    def verify_binding(root,artifact,binding):
        original_verify(root,artifact,binding);verify_recovery(root,workflow,run_id);return True
    def verify_evaluation_smoke(root,folder,plan,bundle,deadline):
        verify_recovery(root,workflow,run_id)
        if plan.get('run_id')!=run_id:raise ValueError('Probe preflight run differs from recovery binding')
        workflow.deadline_guard(deadline)
        result=evaluator.fastprobe_smoke(root,Path(folder)/'probe_preflight')
        workflow.deadline_guard(deadline)
        if result.get('status')!='PASS':raise RuntimeError('Existing convex-probe smoke failed')
        return {**result,'operational_recovery':VERSION,'original_deadline_unix':deadline,
          'stage_plan_sha256':_hash(plan),'registered_probe_policy':copy.deepcopy(plan.get('probe_convex',{})),
          'scientific_source_hashes':copy.deepcopy(plan.get('source_hashes',{})),
          'recovery_runtime_export_sha256':export_hash(Path(root)/RUNTIME),
          'scope':'Existing synthetic solver/resume check only; scientific evaluator and registered probe settings unchanged'}
    verify_evaluation_smoke._operational_recovery_adapter=True
    def bootstrap_source(root):
        code=original_bootstrap(root)
        code+='\n_RECOVERY=types.ModuleType("fmm15_operational_recovery");_RECOVERY.__file__=str(ROOT/'+repr(RUNTIME)+')\n'
        code+='TaggedNotebookLoader(_RECOVERY.__file__).exec_module(_RECOVERY)\n'
        code+='_RECOVERY.install(ROOT,WORKFLOW,'+repr(run_id)+')\n'
        return code
    def spawn(root,artifact,session,expression,path):
        verify_recovery(root,workflow,run_id)
        path=Path(path);new_path=path.with_name(path.stem+'_recovery_v1'+path.suffix)
        return original_spawn(root,artifact,session,expression,new_path)
    workflow.verify_binding=verify_binding;workflow.bootstrap_source=bootstrap_source;workflow.spawn=spawn
    evaluator.verify_evaluation_smoke=verify_evaluation_smoke
    workflow._operational_recovery={'run_id':run_id,'receipt_hash':_hash(receipt)}
    verify_binding(root,artifact,_read(artifact/'study_binding.json'))
    return receipt
